# 01 — Filter PopSign to the SignBridge 20-word vocabulary

Run on Kaggle with the `asl-signs` competition data attached. Outputs `subset.csv` and `split.csv` in `/kaggle/working/`.

In [ ]:
import glob, json, os
import pandas as pd
import numpy as np

# Find the competition folder wherever Kaggle mounted it
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert hits, 'train.csv not found. Attach the asl-signs data: Add Input -> Competitions -> asl-signs'
DATA = os.path.dirname(hits[0])
print('DATA =', DATA)
print(os.listdir(DATA))

In [ ]:
WORDS = ['hello','thankyou','please','bye','yes','no','mom','dad',
         'hungry','drink','potty','owie','sick','hot','sleep',
         'wait','go','finish','mad','sad']

label_map = json.load(open(f'{DATA}/sign_to_prediction_index_map.json'))
missing = [w for w in WORDS if w not in label_map]
assert not missing, f'not in label map: {missing}'
print(f'All {len(WORDS)} words found in {len(label_map)} labels.')

train = pd.read_csv(f'{DATA}/train.csv')
print(train.shape)
train.head()

In [ ]:
sub = train[train['sign'].isin(WORDS)].copy()
print(f'{len(sub)} sequences for {sub.sign.nunique()} words, '
      f'{sub.participant_id.nunique()} participants')

counts = sub['sign'].value_counts().reindex(WORDS)
print(counts.to_string())
print(f'\nmin {counts.min()}  max {counts.max()}  mean {counts.mean():.0f}')

## Signer-independent split

Hold out whole participants, not random clips, so test accuracy means *works on a new person*.

In [ ]:
rng = np.random.default_rng(42)
people = np.array(sorted(sub['participant_id'].unique()))
rng.shuffle(people)
n_test = max(1, round(len(people) * 0.15))
test_people = set(people[:n_test].tolist())

sub['split'] = np.where(sub['participant_id'].isin(test_people), 'test', 'train')
print('test participants:', sorted(test_people))
print(sub['split'].value_counts().to_string())
print()
print(pd.crosstab(sub['sign'], sub['split']).reindex(WORDS).to_string())

## Sequence length

Sample clips to choose the window length (planned: 30 frames).

In [ ]:
sample = sub.sample(min(300, len(sub)), random_state=0)
lens, hand_missing = [], []
for p in sample['path']:
    df = pd.read_parquet(f'{DATA}/{p}', columns=['frame','type','x'])
    lens.append(df['frame'].nunique())
    hands = df[df['type'].isin(['left_hand','right_hand'])]
    # fraction of frames where BOTH hands are entirely NaN
    per_frame = hands.groupby('frame')['x'].apply(lambda s: s.isna().all())
    hand_missing.append(per_frame.mean())

lens = np.array(lens)
print('frames per clip: p10 %d, median %d, p90 %d, max %d' % (
      np.percentile(lens,10), np.median(lens), np.percentile(lens,90), lens.max()))
print('mean fraction of frames with no hand detected: %.2f' % np.mean(hand_missing))

In [ ]:
sub.to_csv('/kaggle/working/subset.csv', index=False)
json.dump({w: i for i, w in enumerate(WORDS)},
          open('/kaggle/working/word_index.json', 'w'), indent=2)
print('saved subset.csv and word_index.json')